# Telco Customer Churn — Feature Engineering

## 1. Objectives & Design Choices
This step transforms business insights identified during EDA into machine-learning-ready features while ensuring interpretability and avoiding data leakage.

**Key principles**
- Features must be business-driven and interpretable
- No target leakage (split before preprocessing)
- Prefer simple transformations over complex feature engineering
- Use a reusable scikit-learn preprocessing pipeline (for consistency across notebooks and Streamlit)

---

## 2. Data Loading & Target Definition
Load the raw dataset using shared project utilities and create a stable binary target (`churn_flag`) for modeling.

---

## 3. Train/Test Split
Split the dataset into training and test sets **before** any preprocessing step to prevent leakage and ensure a reliable evaluation.

---

## 4. Feature Selection
Select features based on business relevance (EDA insights), prioritizing:
- contract type, tenure, pricing, service-related variables

---

## 5. Feature Encoding & Preprocessing
Use a scikit-learn `ColumnTransformer` with:
- imputation + scaling for numerical features
- imputation + one-hot encoding for categorical features

---

## 6. Final Sanity Checks
Validate split sizes, churn rate stability (stratification), and remaining missing values (handled by the preprocessing pipeline).


In [7]:
# ============================================================
# Imports
# Purpose:
# - Keep this notebook focused on feature engineering (no download logic here)
# - Reuse shared utilities to ensure consistency across notebooks and Streamlit
# ============================================================

import sys
from pathlib import Path

# Add project root to PYTHONPATH to allow imports from /src
# (robust when running notebooks from the /notebooks directory)
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np

# Shared project utilities
from src.data_prep import load_telco, get_feature_lists, make_split, build_preprocessor





In [ ]:
# ============================================================
# Load dataset, define features, and build preprocessing pipeline
# Purpose:
# - Load the Telco Customer Churn dataset using shared utilities
# - Define business-driven feature lists
# - Split the data before preprocessing to avoid data leakage
# - Build a reusable preprocessing pipeline for modeling and deployment
# ============================================================

# Load dataset (downloaded from Kaggle only if missing locally)
# This step also:
# - creates a stable binary target variable (churn_flag)
# - applies minimal cleaning (e.g. TotalCharges conversion)
df = load_telco()

# Retrieve feature lists based on business insights
# - cat_features: categorical variables (to be one-hot encoded)
# - num_features: numerical variables (to be imputed and scaled)
cat_features, num_features = get_feature_lists()

# Perform a stratified train/test split
# Split is done BEFORE preprocessing to prevent data leakage
X_train, X_test, y_train, y_test = make_split(df)

# Build the preprocessing pipeline
# - Numerical features: median imputation + standard scaling
# - Categorical features: mode imputation + one-hot encoding
# This pipeline will be reused consistently across:
# - feature engineering
# - modeling
# - Streamlit application
preprocessor = build_preprocessor(cat_features, num_features)

# Quick sanity check
df.head()



Dataset already present in data/raw/


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


# Note: remaining missing values will be handled by the preprocessing pipeline (SimpleImputer)


In [ ]:
# ============================================================
# Data split validation checks
# Purpose:
# - Verify that the train/test split was performed correctly
# - Ensure data integrity before model training
# ============================================================

# Check dataset sizes after the split
# Confirms the expected proportion between training and test sets
print("Train size:", X_train.shape, "Test size:", X_test.shape)

# Compare churn rates between train and test sets
# Stratification should preserve similar churn proportions
print(
    "Churn rate train:", y_train.mean().round(4),
    "test:", y_test.mean().round(4)
)

# Check for remaining missing values in the training features
# Remaining NaNs are expected at this stage and will be handled
# by the preprocessing pipeline (SimpleImputer)
print("Missing values in X_train:", int(X_train.isna().sum().sum()))




Train size: (5634, 19) Test size: (1409, 19)
Churn rate train: 0.2654 test: 0.2654
Missing values in X_train: 8
